# RAG ingestion

Jednorazowy pipeline do dodawania nowego dokumentu miejskiego do bazy Pismaka.

In [1]:
from pathlib import Path
import os, sys, json

PROJECT_ROOT = Path(r"C:\Krzysiek\Pismak")
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from Zrodlowyszukiwator.rag_ingestion.catalog import load_catalog
from Zrodlowyszukiwator.rag_ingestion.pipeline import ingest_pdf
from Zrodlowyszukiwator.rag_ingestion.embedder import OllamaEmbedder
from Zrodlowyszukiwator.rag_ingestion.repository import PgVectorRepository

print("Gotowe.")

Gotowe.


In [2]:
# Konfiguracja.
# Wskaż jeden plik dodawany w tej sesji.

POLICIES_DIR = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "RAW"
CATALOG_PATH = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "document_catalog.json"
POLICIES_LIST = list(POLICIES_DIR.iterdir())
POLICIES_LIST

[WindowsPath('C:/Krzysiek/Pismak/src/Zrodlowyszukiwator/RAG/RAW/00_Strategia-rozwoju-Wroclawia-Wroclaw-2050.pdf'),
 WindowsPath('C:/Krzysiek/Pismak/src/Zrodlowyszukiwator/RAG/RAW/00_Strategia_2030.pdf'),
 WindowsPath('C:/Krzysiek/Pismak/src/Zrodlowyszukiwator/RAG/RAW/01_Polityka-rowerowa-wroclawia.pdf'),
 WindowsPath('C:/Krzysiek/Pismak/src/Zrodlowyszukiwator/RAG/RAW/01_Wroclawska_polityka_mobilnosci.pdf'),
 WindowsPath('C:/Krzysiek/Pismak/src/Zrodlowyszukiwator/RAG/RAW/02_Wrocławski_program_tramwajowy_2.0.pdf'),
 WindowsPath('C:/Krzysiek/Pismak/src/Zrodlowyszukiwator/RAG/RAW/03_Wroclawskie_standardy_dostepnosci_przestrzeni_miejskich.pdf')]

In [3]:
catalog = load_catalog(CATALOG_PATH)

In [4]:
DATABASE_URL = os.environ["DATABASE_URL"]
EMBEDDING_DIM = int(os.getenv("RAG_EMBEDDING_DIM", "1024"))

repo = PgVectorRepository(
    DATABASE_URL,
    embedding_dim=EMBEDDING_DIM,
)
repo.ensure_schema()

print("Schema gotowy.")

Schema gotowy.


In [5]:
embedder = OllamaEmbedder(
    model=os.getenv("RAG_EMBEDDING_MODEL", "bge-m3"),
    base_url=os.getenv("OLLAMA_URL", "http://127.0.0.1:11434"),
    batch_size=16,
)

test_embedding = embedder.embed(["Test embeddingu dla dokumentów miejskich."])[0]
print("Wymiar embeddingu:", len(test_embedding))

if len(test_embedding) != EMBEDDING_DIM:
    raise ValueError(
        f"Model zwraca {len(test_embedding)} wymiarów, "
        f"a DB jest skonfigurowana na {EMBEDDING_DIM}."
    )

Wymiar embeddingu: 1024


In [6]:
# wrzuć wszystkie do bazy danych.

for file in POLICIES_LIST:
    if file.suffix.lower() == ".pdf":
        print(f"Znaleziono plik PDF: {file.name}")


        if file.name not in catalog:
            raise KeyError(
                f"Brak metadanych dla {file.name!r} w {CATALOG_PATH}"
            )

        metadata = catalog[file.name]
        display(metadata)

        result = ingest_pdf(
            file,
            metadata,
            repository=repo,
            embedder=embedder,
            max_chars=3500,
            min_chars=250,
            overlap_units=1,
            force=False,
        )

result

Znaleziono plik PDF: 00_Strategia-rozwoju-Wroclawia-Wroclaw-2050.pdf


DocumentMetadata(document_id='wroclaw-strategy-2050-2025', title='Strategia Rozwoju Wrocławia „Wrocław 2050”', document_kind=<DocumentKind.STRATEGY: 'strategy'>, city='Wrocław', issuer='Gmina Wrocław', adopting_body='Rada Miejska Wrocławia', legal_form=<LegalForm.RESOLUTION: 'resolution'>, adoption_act_number='XXIV/463/25', adoption_date=datetime.date(2025, 10, 20), valid_from=datetime.date(2025, 10, 20), valid_to=None, is_active=True, source_url=None, language='pl', tags=['strategia', 'rozwój miasta', 'Wrocław 2050'], relations=[], notes='Dokument nadrzędny dla polityk, programów i planów działań w hierarchii merytorycznej RAG.')

Znaleziono plik PDF: 00_Strategia_2030.pdf


DocumentMetadata(document_id='wroclaw-strategy-2030', title='Strategia Rozwoju Wrocławia „Wrocław 2030”', document_kind=<DocumentKind.STRATEGY: 'strategy'>, city='Wrocław', issuer='Gmina Wrocław', adopting_body='Rada Miejska Wrocławia', legal_form=<LegalForm.RESOLUTION: 'resolution'>, adoption_act_number='XXIV/463/25', adoption_date=datetime.date(2025, 10, 20), valid_from=datetime.date(2017, 1, 1), valid_to=datetime.date(2025, 10, 20), is_active=False, source_url=None, language='pl', tags=['strategia', 'rozwój miasta', 'Wrocław 2030'], relations=[], notes='Dokument nadrzędny dla polityk, programów i planów działań w hierarchii merytorycznej RAG.')

Znaleziono plik PDF: 01_Polityka-rowerowa-wroclawia.pdf


...

Znaleziono plik PDF: 02_Wrocławski_program_tramwajowy_2.0.pdf


DocumentMetadata(document_id='wroclaw-tram-program-2-0', title='Wrocławski Program Tramwajowy 2.0 — raport', document_kind=<DocumentKind.PROGRAM: 'program'>, city='Wrocław', issuer='Urząd Miejski Wrocławia', adopting_body=None, legal_form=<LegalForm.REPORT: 'report'>, adoption_act_number=None, adoption_date=None, valid_from=None, valid_to=None, is_active=None, source_url=None, language='pl', tags=['tramwaje', 'transport zbiorowy', 'komunikacja miejska', 'inwestycje', 'program'], relations=[DocumentRelation(target_document_id='wroclaw-mobility-policy-2013', relation='references', note='Program wprost nawiązuje do celów Wrocławskiej Polityki Mobilności.')], notes='Materiał analityczny wspomagający decyzje inwestycyjne; umieszczony między polityką a standardami w hierarchii merytorycznej.')

Znaleziono plik PDF: 03_Wroclawskie_standardy_dostepnosci_przestrzeni_miejskich.pdf


DocumentMetadata(document_id='wroclaw-accessibility-standards', title='Wrocławskie standardy dostępności przestrzeni miejskich', document_kind=<DocumentKind.STANDARD: 'standard'>, city='Wrocław', issuer='Gmina Wrocław / Miejski Ośrodek Pomocy Społecznej we Wrocławiu', adopting_body=None, legal_form=<LegalForm.STANDALONE: 'standalone'>, adoption_act_number=None, adoption_date=None, valid_from=None, valid_to=None, is_active=None, source_url=None, language='pl', tags=['dostępność', 'projektowanie uniwersalne', 'przestrzeń publiczna', 'standardy'], relations=[], notes='Dokument wytycznych; tekst wskazuje obowiązek stosowania przez jednostki organizacyjne miasta.')

{'status': 'ingested',
 'document_id': 'wroclaw-accessibility-standards',
 'title': 'Wrocławskie standardy dostępności przestrzeni miejskich',
 'pages': 129,
 'chunks': 503,
 'low_text_pages': [1, 2],
 'sha256': 'd3e86cc9332b929a3939e0cd119e2eed20970fac9efcb01fa90c7d53fed4552b',
 'embedding_dim': 1024}

In [7]:
# Kontrola bazy.

doc_count, chunk_count = repo.counts()

print("Dokumenty w bazie:", doc_count)
print("Chunki w bazie:", chunk_count)

Dokumenty w bazie: 6
Chunki w bazie: 1385
